# STEP 05 — 모델 학습 및 최종 평가

**목적:** Logistic Regression과 Random Forest를 `GridSearchCV`로 최적화하고, **5-Fold 훈련 교차검증 F1-score**를 기준으로 최적 조합을 선정한 뒤 홀드아웃 테스트 데이터에서 Accuracy / Precision / Recall / F1 / ROC-AUC를 측정한다.

> 이 파일은 **STEP 05만** 수행하는 **독립 실행 노트북**이다. STEP 04 파일을 먼저 실행하거나 결과 CSV를 불러올 필요가 없다. 대신 동일한 `seed=42`, 학습·테스트 분할, 파생 변수, 전처리/변수 선택 정의를 포함한다.
>
> **주의:** 테스트 데이터는 그리드 탐색·특성 선택에 이용하지 않는다. 최종 평가 점수를 보고 하이퍼파라미터를 다시 바꾸지 않는다.

## 1. 데이터 로드 및 고정된 분할

In [ ]:
# 패키지 설치: %pip install pandas numpy matplotlib scikit-learn joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 42
CSV_PATH = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f'{CSV_PATH} 파일을 찾지 못했습니다. 노트북이 들어있는 프로젝트 폴더에서 실행하고 '
        '필수 입력 파일: data/WA_Fn-UseC_-Telco-Customer-Churn.csv.'
    )

df = pd.read_csv(CSV_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
X = df.drop(columns=['customerID','Churn'])
y = df['Churn'].map({'No':0, 'Yes':1})
if y.isna().any():
    raise ValueError('Churn에 허용 범주 No/Yes 외의 값이 포함되어 있다.')
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
OUT_DIR = Path('results')
OUT_DIR.mkdir(exist_ok=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('전체:', df.shape, '| 학습:', X_train.shape, '| 테스트:', X_test.shape)
print('학습 이탈률:', f'{y_train.mean():.2%}', '| 테스트 이탈률:', f'{y_test.mean():.2%}')
print('TotalCharges 결측:', df['TotalCharges'].isna().sum())

## 2. 공통 Pipeline 정의 (STEP 04와 동일한 조건)

In [ ]:
SERVICE_COLUMNS = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                   'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
NUMERIC_COLS = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges',
                'ServiceCount','AvgChargeRatio']
CATEGORY_COLS = ['gender','Partner','Dependents','PhoneService','MultipleLines',
                 'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
                 'TechSupport','StreamingTV','StreamingMovies','Contract',
                 'PaperlessBilling','PaymentMethod','TenureGroup']

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """각 행의 값으로만 파생 변수를 만들며, 목표값 Churn은 사용하지 않음."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        frame = X.copy()
        frame['TenureGroup'] = pd.cut(
            frame['tenure'], [-1,12,24,48,np.inf],
            labels=['0-12m','13-24m','25-48m','49m+']
        ).astype('object')
        frame['ServiceCount'] = sum(
            frame[col].eq('Yes').astype(int) for col in SERVICE_COLUMNS
        )
        denom = frame['MonthlyCharges'] * frame['tenure']
        frame['AvgChargeRatio'] = (
            frame['TotalCharges'] / denom.replace(0, np.nan)
        ).replace([np.inf,-np.inf], np.nan)
        return frame

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:  # scikit-learn 1.2 이전 호환
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), NUMERIC_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', make_ohe())
        ]), CATEGORY_COLS)
    ], sparse_threshold=0)

class RFTopKSelector(BaseEstimator, TransformerMixin):
    """훈련 fold에서만 RF 중요도를 학습하는 Top-K 특성 선택기."""
    def __init__(self, k='all', n_estimators=40, random_state=42):
        self.k = k
        self.n_estimators = n_estimators
        self.random_state = random_state

    def fit(self, X, y):
        if self.k == 'all':
            self.indices_ = np.arange(X.shape[1])
        else:
            forest = RandomForestClassifier(
                n_estimators=self.n_estimators, random_state=self.random_state,
                class_weight='balanced', n_jobs=1
            )
            forest.fit(X, y)
            self.importances_ = forest.feature_importances_
            k = min(int(self.k), X.shape[1])
            self.indices_ = np.argsort(self.importances_)[::-1][:k]
        return self

    def transform(self, X):
        return X[:, self.indices_]

def make_pipeline(model, k='all'):
    return Pipeline([
        ('features', FeatureEngineer()),
        ('preprocess', make_preprocessor()),
        ('select', RFTopKSelector(k=k, random_state=SEED)),
        ('model', model)
    ])

print('Pipeline 단계: features -> preprocess -> select -> model')

## 3. GridSearchCV — 변수 수 + 하이퍼파라미터 탐색

- Logistic Regression: 변수 수 `all/10`, 정규화 강도 `C = 0.1/1.0`
- Random Forest: 변수 수 `all/10`, `min_samples_leaf = 1/3`
- 모든 파라미터 조합은 **훈련 데이터 5-Fold CV**로만 평가한다.
- 최적화 지표는 **F1-score**다. ROC-AUC 등은 보조 지표다.

계산 비용을 고려하여 모델별 4개 하이퍼파라미터 조합으로 탐색 범위를 제한하였다.

In [ ]:
from sklearn.model_selection import GridSearchCV

SCORING = {'F1':'f1', 'ROC-AUC':'roc_auc', 'Accuracy':'accuracy',
           'Precision':'precision', 'Recall':'recall'}
SEARCH_SPECS = {
    'Logistic Regression': (
        LogisticRegression(max_iter=3000, random_state=SEED),
        {'select__k':['all',10], 'model__C':[0.1,1.0]}
    ),
    'Random Forest': (
        RandomForestClassifier(
            n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=1
        ),
        {'select__k':['all',10], 'model__min_samples_leaf':[1,3]}
    )
}

searches = {}
cv_tables = []
for name, (model, grid) in SEARCH_SPECS.items():
    print('GridSearchCV:', name)
    search = GridSearchCV(
        estimator=make_pipeline(model), param_grid=grid,
        scoring=SCORING, refit='F1', cv=CV, n_jobs=1,
        return_train_score=True, error_score='raise'
    )
    search.fit(X_train, y_train)
    searches[name] = search
    results = pd.DataFrame(search.cv_results_)
    results.insert(0, '모델', name)
    cv_tables.append(results)
    print('  선택된 파라미터:', search.best_params_)
    print('  최적 CV F1:', round(search.best_score_, 4))

cv_results = pd.concat(cv_tables, ignore_index=True)
cols = ['모델','param_select__k','mean_train_F1','mean_test_F1',
        'std_test_F1','mean_test_ROC-AUC','params']
display(cv_results[cols].sort_values(['모델','mean_test_F1'],
                                     ascending=[True,False]).round(4))
cv_results[cols].to_csv(OUT_DIR/'step05_gridsearch_cv_results.csv',
                        index=False, encoding='utf-8-sig')

## 4. CV 기준 최적 모델 결정 (테스트 사용 전)

각 모델의 최고 F1 조합을 비교해 **대표 모델 1개**를 선정한다. 모델 비교 결과를 고른 뒤에만 테스트 데이터 평가를 진행한다.

In [ ]:
selection_df = pd.DataFrame([
    {'모델':name,
     '선택 변수 수':str(search.best_params_['select__k']),
     'CV F1':search.best_score_,
     '최적 파라미터':str(search.best_params_)}
    for name, search in searches.items()
]).sort_values('CV F1', ascending=False).reset_index(drop=True)
WINNER_NAME = selection_df.loc[0, '모델']
display(selection_df.round(4))
selection_df.to_csv(OUT_DIR/'step05_cv_selected_models.csv',
                    index=False,encoding='utf-8-sig')
print('최종 대표 모델 (훈련 CV F1 기준):', WINNER_NAME)

## 5. 선정된 모델들의 최종 테스트 성능 비교

각 모델에서 **CV로 선정한 하나의 조합**을 테스트 데이터에 적용한다. 두 모델의 최종 테스트 성능은 비교하되, 테스트 점수가 더 좋았다는 이유만으로 모델·변수 수·임곗값을 다시 고르지 않는다.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

test_rows = []
for name, search in searches.items():
    pipe = search.best_estimator_  # 전체 학습 데이터에 refit된 Pipeline
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:,1]
    test_rows.append({
        '모델':name, '선택 변수 수':str(search.best_params_['select__k']),
        'CV F1':search.best_score_,
        'Accuracy':accuracy_score(y_test,y_pred),
        'Precision':precision_score(y_test,y_pred,zero_division=0),
        'Recall':recall_score(y_test,y_pred,zero_division=0),
        'F1':f1_score(y_test,y_pred,zero_division=0),
        'ROC-AUC':roc_auc_score(y_test,y_prob)
    })

test_df = pd.DataFrame(test_rows)
display(test_df.round(4))
test_df.to_csv(OUT_DIR/'step05_final_test_metrics.csv',
               index=False,encoding='utf-8-sig')
print('대표 모델:', WINNER_NAME)
print(classification_report(
    y_test, searches[WINNER_NAME].best_estimator_.predict(X_test),
    target_names=['유지(No)','이탈(Yes)'],zero_division=0
))

## 6. ROC Curve 및 Confusion Matrix

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

fig, axes = plt.subplots(1,2,figsize=(12,5))
for name, search in searches.items():
    RocCurveDisplay.from_estimator(search.best_estimator_, X_test, y_test,
                                   ax=axes[0],name=name)
axes[0].set_title('STEP 05: ROC Curves (CV-selected models)')
axes[0].grid(alpha=0.2)
ConfusionMatrixDisplay.from_estimator(
    searches[WINNER_NAME].best_estimator_, X_test, y_test,
    ax=axes[1], display_labels=['No','Yes'], cmap='Blues', colorbar=False
)
axes[1].set_title('Confusion Matrix: '+WINNER_NAME)
plt.tight_layout()
plt.savefig(OUT_DIR/'step05_roc_confusion_matrix.png',dpi=170,bbox_inches='tight')
plt.show()

## 7. 최종 모델에 선택된 특성 및 파이프라인 저장

`joblib`로 저장한 모델을 불러올 때는 **신뢰할 수 있는 파일만** 사용한다. 사용자 정의 `FeatureEngineer`, `RFTopKSelector` 클래스가 있는 환경에서 로딩해야 한다.

In [ ]:
import joblib

for name, search in searches.items():
    pipe = search.best_estimator_
    names = pipe.named_steps['preprocess'].get_feature_names_out()
    indices = pipe.named_steps['select'].indices_
    chosen = list(names[indices])
    print(name, '| CV 선택 변수 수:', len(chosen))
    print(chosen if len(chosen)<=20 else '전체 변수 사용')
    filename = 'step05_model_'+name.lower().replace(' ','_')+'.joblib'
    joblib.dump(pipe, OUT_DIR/filename)

print('STEP 05 완료! 저장 폴더:',OUT_DIR.resolve())

## 평가 결과의 해석 범위

- 두 모델에 동일한 계층 분할과 전처리 조건을 적용한다.
- GridSearchCV는 훈련 내부 5-Fold F1로 설정을 선택한다.
- Precision, Recall, F1은 이탈 클래스를 기준으로 계산한다.
- 테스트셋은 STEP03에서 탐색한 분할의 재사용 결과이며 독립적인 최종 검증 점수와 구분한다.
- 중첩 CV는 `robust_evaluation.py`에서 수행한다. 선행 데이터 탐색의 영향은 여전히 남는다.

**저장 파일:** `step05_gridsearch_cv_results.csv`, `step05_cv_selected_models.csv`, `step05_final_test_metrics.csv`, `step05_roc_confusion_matrix.png`, `step05_model_*.joblib`